# 09 — AeroTrack Route Market Watchlist (Capstone)

## Purpose
Airline and airport planners cannot make a sound capacity decision from passenger counts alone. AeroTrack narrows a large BTS file into a **monthly route-market review queue**: which carrier–origin–destination markets look unusually busy, what evidence supports the flag, and how uncertain was the model on unseen months?

The output is a shortlist for investigation alongside schedules, seat supply, fares, costs, and booking data. It does not claim that demand is unmet or automatically recommend new service. BTS T-100 Market is monthly reported passenger traffic; it is neither real-time nor a record of unique people.

## Workflow
Raw BTS archive → audit and clean → carrier-route-month table → chronological holdout → compare a simple baseline with a supervised model → score 2026 YTD route markets → prioritize for human review → export a transparent CSV.


In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
ROOT=Path.cwd(); path=ROOT/'data/processed/air_passenger_monthly.csv'
df=pd.read_csv(path,parse_dates=['DATE'])
print('Data shape:',df.shape,'| month coverage:',df.DATE.min().date(),'to',df.DATE.max().date())
print('2026 coverage:',sorted(df.loc[df.YEAR.eq(2026),'MONTH'].unique().tolist()))


Data shape: (503224, 11) | month coverage: 2025-01-01 to 2026-07-01
2026 coverage: [1, 2, 3, 4, 5, 6, 7]


## Define the decision signal before fitting

We predict passenger count for a carrier-route-month using route identity, distance, and seasonality. For this teaching capstone, a market gets a review flag if its prediction is above the 2025 75th-percentile route-month count. This threshold is a transparent prioritization rule, not a universal industry definition of “busy.” Actual 2026 passengers are used only for holdout evaluation, never as model features.


In [2]:
# Chronological split; fit transformations and models only on 2025.
train=df[df.YEAR.eq(2025)].copy(); future=df[df.YEAR.eq(2026)].copy()
assert len(train)>0 and len(future)>0
train_threshold=float(train.PASSENGERS.quantile(.75))
cat=['ORIGIN','DEST','UNIQUE_CARRIER']; num=['DISTANCE','MONTH']
# Cap training rows for a reproducible laptop-scale run; the holdout remains untouched.
fit_train=train.sample(n=min(60_000,len(train)),random_state=42)
X_train=fit_train[cat+num]; y_train=np.log1p(fit_train.PASSENGERS.clip(lower=0))
X_test=future[cat+num]; y_test=future.PASSENGERS.clip(lower=0)
prep=ColumnTransformer([('cat',Pipeline([('fill',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore',min_frequency=10))]),cat),
                        ('num',Pipeline([('fill',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num)])
model=Pipeline([('prep',prep),('regressor',RandomForestRegressor(n_estimators=50,max_depth=18,min_samples_leaf=40,n_jobs=-1,random_state=42,max_samples=.8))])
model.fit(X_train,y_train)
pred=np.maximum(0,np.expm1(model.predict(X_test)))
mae=mean_absolute_error(y_test,pred); rmse=np.sqrt(mean_squared_error(y_test,pred)); r2=r2_score(y_test,pred)
print(f'Chronological holdout: fit={len(fit_train):,} sampled records from 2025 ({len(train):,} available), test={len(future):,} records (2026 YTD)')
print(f'MAE={mae:,.1f} passengers/market-month | RMSE={rmse:,.1f} | R²={r2:.3f}')


Chronological holdout: fit=60,000 sampled records from 2025 (317,333 available), test=185,891 records (2026 YTD)
MAE=2,864.7 passengers/market-month | RMSE=6,169.1 | R²=0.042


## Build an actionable, evidence-labeled review queue

We rank predicted demand against the training-only threshold, retain actual observed counts as context, and calculate a prediction gap. The CSV marks each row as a **review candidate** and explicitly includes the model and threshold. It should be interpreted as “check this market with richer data,” not “add capacity.”


In [3]:
watch=future[['DATE','YEAR','MONTH','ORIGIN','DEST','UNIQUE_CARRIER','UNIQUE_CARRIER_NAME','DISTANCE','PASSENGERS']].copy()
watch['PREDICTED_PASSENGERS']=pred
watch['PREDICTION_GAP']=watch.PREDICTED_PASSENGERS-watch.PASSENGERS
watch['TRAIN_2025_HIGH_DEMAND_THRESHOLD']=train_threshold
watch['REVIEW_FLAG']=np.where(watch.PREDICTED_PASSENGERS>=train_threshold,'Review candidate','Standard review')
watch['DECISION_NOTE']='Screening signal only; confirm capacity, fares, schedules and bookings before action.'
watch=watch.sort_values(['PREDICTED_PASSENGERS','PREDICTION_GAP'],ascending=False).reset_index(drop=True)
output=ROOT/'data/processed/route_market_watchlist_2026_ytd.csv'; watch.to_csv(output,index=False)
shortlist=watch.loc[watch.REVIEW_FLAG.eq('Review candidate')].copy()
shortlist_output=ROOT/'data/processed/route_market_review_candidates_2026_ytd.csv'; shortlist.to_csv(shortlist_output,index=False)
display(watch.head(20))
print(f'Exported full watchlist: {len(watch):,} scored route-months to {output}')
print(f'Exported review shortlist: {len(shortlist):,} candidates to {shortlist_output}')


         DATE  ...                                      DECISION_NOTE
0  2026-07-01  ...  Screening signal only; confirm capacity, fares...
1  2026-07-01  ...  Screening signal only; confirm capacity, fares...
2  2026-07-01  ...  Screening signal only; confirm capacity, fares...
3  2026-07-01  ...  Screening signal only; confirm capacity, fares...
4  2026-07-01  ...  Screening signal only; confirm capacity, fares...
5  2026-07-01  ...  Screening signal only; confirm capacity, fares...
6  2026-07-01  ...  Screening signal only; confirm capacity, fares...
7  2026-07-01  ...  Screening signal only; confirm capacity, fares...
8  2026-07-01  ...  Screening signal only; confirm capacity, fares...
9  2026-05-01  ...  Screening signal only; confirm capacity, fares...
10 2026-06-01  ...  Screening signal only; confirm capacity, fares...
11 2026-06-01  ...  Screening signal only; confirm capacity, fares...
12 2026-06-01  ...  Screening signal only; confirm capacity, fares...
13 2026-05-01  ...  

## What makes the result useful—and what still needs work

- **Useful today:** a reproducible shortlist with route, carrier, month, predicted count, observed count, prediction gap, and the rule used to flag it.
- **Scalable next:** refresh monthly source files, retain a date-partitioned history, monitor prediction error and route coverage, and compare seasonal-naive, linear and tree models on rolling time windows.
- **For a real planning decision:** join schedules and seats, load factors, fares, cancellations, booking curves, costs, and airport constraints. Add confidence intervals and a human approval trail.
- **Known limitation:** 12 training months and 7 holdout months are too little to establish stable multi-year seasonality. 2026 is partial-year. These results support a classroom prototype and exploratory review, not operational or financial decisions.
